<a href="https://colab.research.google.com/github/yc-115/programing-language/blob/main/%E3%80%8CHW2_%E6%88%90%E7%B8%BE%E4%B8%80%E6%9C%AC%E9%80%9A_Part1_ipynb%E3%80%8D41371211H.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
!pip install -q google-generativeai

In [13]:
import gradio as gr
import pandas as pd
from google.colab import auth
from google.auth import default

# -*- coding: utf-8 -*-
import gspread
from datetime import datetime
import google.generativeai as genai
import os
import json

In [14]:
from google.colab import userdata
from google import genai

# 從 Colab Secrets 中獲取 API 金鑰
api_key = userdata.get('gemini')

# 使用獲取的金鑰配置 genai
client = genai.Client(api_key=api_key)

MODEL_ID = 'gemini-2.5-flash'

In [15]:
response = client.models.generate_content(
    model = MODEL_ID, contents="Explain how AI works in a few words"
)
print(response.text)

It learns patterns from data to act intelligently.


In [16]:
SHEET_URL = "https://docs.google.com/spreadsheets/d/1HfkscKzGxU9L_tcDrdEaCFi3xN_Gttih-MPP91rN7t0/edit?usp=sharing"
WORKSHEET_NAME = "工作表2"

REQUIRED_COLUMNS = ["日期", "考試名稱", "科目", "作業成績"]

_auth_done = False
_gc = None
_ws = None

In [17]:
# --- 主要功能區塊 ---
def get_user_grades():
    """
    透過終端機輸入學生成績，直到使用者輸入 'q' 結束。
    """
    print("--- 準備輸入成績。輸入 'q' 來停止。---")
    grades = []

    exam_name = input("請輸入考試名稱（例如：第一次段考、期中考）：")
    if not exam_name:
        print("考試名稱不能為空，將使用預設名稱 '一般評量'。")
        exam_name = "一般評量"

    while True:
        subject = input("請輸入科目（或輸入 'q' 停止）：")
        if subject.lower() == 'q':
            break

        grade = input(f"請輸入 {subject} 的成績：")
        try:
            grade = int(grade)
        except ValueError:
            print("成績必須是數字。請重新輸入。")
            continue

        today = datetime.now().strftime('%Y-%m-%d')
        grades.append([today, exam_name, subject, grade])
        print(f"已記錄：日期: {today}, 考試名稱: {exam_name}, 科目: {subject}, 成績: {grade}\n")

    return grades

In [18]:
def get_ai_summary(grades):
    """
    呼叫 Gemini 模型來生成成績摘要與常見迷思。
    """
    # 準備給 AI 的提示
    prompt_text = "以下是學生的成績列表，請幫我根據這些成績，產出一個簡單的摘要與常見迷思整理（不評分，只做總結）。\n\n"
    for record in grades:
        date, exam_name, subject, grade = record
        prompt_text += f"日期：{date}, 考試名稱：{exam_name}, 科目：{subject}, 成績：{grade}\n"

    print("\n--- 正在呼叫 AI 模型生成摘要... ---")
    try:
        response = client.models.generate_content(model = MODEL_ID, contents = prompt_text)
        summary = response.text
        return summary
    except Exception as e:
        print(f"呼叫 AI 時發生錯誤：{e}")
        return "AI 摘要生成失敗。"

In [8]:
new_grades = get_user_grades()

--- 準備輸入成績。輸入 'q' 來停止。---
請輸入科目（或輸入 'q' 停止）：國文
請輸入 國文 的成績：85
已記錄：日期: 2026-04-07, 科目: 國文, 成績: 85

請輸入科目（或輸入 'q' 停止）：q


In [9]:
new_grades

[['2026-04-07', '國文', 85]]

In [10]:
get_ai_summary(new_grades)


--- 正在呼叫 AI 模型生成摘要... ---


'好的，針對您提供的學生國文成績，以下是簡單的摘要與常見迷思整理：\n\n---\n\n### 學生成績摘要\n\n根據您提供的資訊，該學生在**2026年4月7日的國文科目中獲得了85分**。\n\n*   **科目：** 國文\n*   **日期：** 2026年4月7日\n*   **分數：** 85分\n\n此分數落在1到100分的評分範圍內，代表學生在該次評量中對國文內容的掌握程度。\n\n---\n\n### 成績評量常見迷思整理 (不評分，只做總結)\n\n在看待學生的成績時，我們常常會有一些直覺性的判斷，但這些判斷有時可能導致誤解。以下整理幾個與成績評量相關的常見迷思：\n\n1.  **迷思一：單一成績能完全代表學生的學習全貌或能力。**\n    *   **事實：** 單一成績僅反映學生在特定時間點、特定評量方式下，對特定範圍內容的掌握程度。它無法全面呈現學生的學習過程、努力程度、思考深度、多元能力（如創造力、合作能力、解決問題能力）或在其他情境下的表現。學生的整體學習表現是動態且多面向的。\n\n2.  **迷思二：成績高低直接等於努力程度。**\n    *   **事實：** 努力是影響成績的關鍵因素之一，但成績也受其他多重因素影響，例如：學習策略的有效性、先備知識、科目難度、考試設計、個人學習風格、甚至當天身體狀況、焦慮程度等。有些學生學習效率高，看似不費力卻能取得好成績；也可能學生非常努力，但若方法不對或未掌握重點，成績也可能不如預期。\n\n3.  **迷思三：成績是衡量學生價值或未來成功的唯一標準。**\n    *   **事實：** 成績是學業表現的量化指標，但學生的價值與潛力遠超出數字。品格、創造力、解決問題能力、溝通協調能力、團隊合作、情緒智商、批判性思維、對學習的熱情等，都是成績無法量化的重要特質，卻對個人成長與未來發展至關重要。過度強調成績可能忽略了學生的其他優勢與發展潛力。\n\n4.  **迷思四：所有的分數都是可直接比較的。**\n    *   **事實：** 即使是同一科目，不同評量方式（例如：選擇題、申論題、實作題、口語報告、專題報告）、不同考試範圍、不同難易度的試卷，其分數所代表的意義可能不同。將不同情境、不同標準下的分數直接比較，可能會產生誤導，且無法提供有意義的學習回饋。\n\n5.  **迷思五：高

In [19]:
def main():
    """
    主程式流程：輸入成績 -> 獲取 AI 摘要 -> 寫入 Google Sheet。
    """
    try:
        # 1. Google Sheet 身份驗證
        auth.authenticate_user()

        creds, _ = default()
        gc = gspread.authorize(creds)

        sh = gc.open_by_url(SHEET_URL)
        ws = sh.worksheet(WORKSHEET_NAME)

        print("--- Google Sheet 連線成功。---")

        # 2. 獲取使用者輸入的成績
        new_grades = get_user_grades()

        if not new_grades:
            print("沒有輸入任何成績，程式結束。")
            return

        # 3. 將新成績寫入 Google Sheet
        ws.append_rows(new_grades)
        print("\n--- 成績已成功寫入 Google Sheet。---")

        # 4. 獲取 AI 摘要並寫入 Google Sheet
        summary = get_ai_summary(new_grades)

        # 獲取用於 AI 摘要的考試名稱 (從第一筆成績記錄中取得)
        summary_exam_name = new_grades[0][1] if new_grades else "多科目總結"

        # 尋找第一行空白列作為 AI 摘要的起始位置
        next_row_for_summary = len(ws.col_values(1)) + 1

        # 使用 update_cell() 方法更新儲存格
        ws.update_cell(next_row_for_summary, 1, datetime.now().strftime('%Y-%m-%d')) # 日期
        ws.update_cell(next_row_for_summary, 2, summary_exam_name) # 考試名稱
        ws.update_cell(next_row_for_summary, 3, 'AI 摘要') # 摘要標籤

        # 為了避免單元格內容過長，將摘要內容分成多行來寫入，從第四個欄位開始
        summary_lines = summary.split('\n')
        for i, line in enumerate(summary_lines):
            ws.update_cell(next_row_for_summary + i, 4, line)

        print("\n--- AI 摘要已成功寫入 Google Sheet。---")
        print("以下是 AI 生成的摘要內容：")
        print("-" * 50)
        print(summary)
        print("-" * 50)

    except gspread.exceptions.APIError as e:
        print(f"Google Sheets API 錯誤：{e.response.text}")
        print("請確認：")
        print("1. 您的服務帳戶金鑰檔案正確且未過期。")
        print("2. 您已將服務帳戶的 Email 地址（在 JSON 檔案中）分享給 Google Sheet，並給予編輯權限。")
    except Exception as e:
        print(f"發生未預期的錯誤：{e}")

if __name__ == "__main__":
    main()

--- Google Sheet 連線成功。---
--- 準備輸入成績。輸入 'q' 來停止。---
請輸入考試名稱（例如：第一次段考、期中考）：開學考
請輸入科目（或輸入 'q' 停止）：國文
請輸入 國文 的成績：75
已記錄：日期: 2026-04-07, 考試名稱: 開學考, 科目: 國文, 成績: 75

請輸入科目（或輸入 'q' 停止）：英文
請輸入 英文 的成績：88
已記錄：日期: 2026-04-07, 考試名稱: 開學考, 科目: 英文, 成績: 88

請輸入科目（或輸入 'q' 停止）：數學
請輸入 數學 的成績：100
已記錄：日期: 2026-04-07, 考試名稱: 開學考, 科目: 數學, 成績: 100

請輸入科目（或輸入 'q' 停止）：q

--- 成績已成功寫入 Google Sheet。---

--- 正在呼叫 AI 模型生成摘要... ---

--- AI 摘要已成功寫入 Google Sheet。---
以下是 AI 生成的摘要內容：
--------------------------------------------------
好的，這是一份根據您提供的成績資料所做的簡單摘要與常見迷思整理，其中不包含任何評分或評價：

---

### **成績摘要**

這份成績資料顯示學生在2026年04月07日參加了一次名為「開學考」的考試。考試科目包含國文、英文和數學，各科的成績如下：

*   **國文：** 75分
*   **英文：** 88分
*   **數學：** 100分

整體而言，這次考試的科目成績分佈從75分到100分之間。

---

### **關於成績的常見迷思整理**

在看待學生成績時，我們常會有一些常見的誤解，以下將其整理說明，這些迷思的澄清有助於我們以更全面和健康的視角來看待學習歷程：

1.  **迷思一：成績是衡量一個人全部價值的唯一標準。**
    *   **澄清：** 成績主要反映學生在特定時間、特定科目上對知識的掌握程度。它無法完全衡量一個人的創意、情商、解決問題的能力、人際關係、努力程度、潛力或個人品格。將成績等同於一個人的全部價值，容易忽略學生的多樣性與其他重要特質。

2.  **迷思二：單一考試的成績能完全定義學生的學習狀況。**
    *   **澄清：